# End of week 1 exercise

To demonstrate your familiarity with OpenAI API, and also Ollama, build a tool that takes a technical question,  
and responds with an explanation. This is a tool that you will be able to use yourself during the course!

In [16]:
# imports
import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [17]:
# constants

MODEL_GPT = 'gpt-5-nano'
MODEL_LLAMA = 'llama3.2:1b'

OLLAMA_BASE_URL = "http://localhost:11434/v1"

ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')

In [18]:
# set up environment
load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
openai = OpenAI()

system_prompt = """
You are a senior software engineer with the ability to answer and simplify technical explanations to any question you are asked as if you are explaining to a 5 years old kid.
You should respond in Mardksow with no codes included and use diagrams if needed 
        
"""

# {
#     "question": "What is the difference between HTTP and HTTPS ?",
#     "answer": "HTTP is like sending a postcard that anyone can read, while HTTPS is like sending a message inside a locked box that keeps your information safe by encrypting it, making HTTPS the safer way to communicate with websites."
# }

API key looks good so far


In [19]:
# here is the question; type over this to ask something new

question = """
Please explain what this code does and why:
yield from {book.get("author") for book in books if book.get("author")}
"""

def get_client(model):
    return ollama if model == MODEL_LLAMA else openai

def answer_question(modelToAnswer, question):
    stream = get_client(modelToAnswer).chat.completions.create(
        model=modelToAnswer,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": question}
        ],
        stream=True
    )

    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

answer_question(MODEL_GPT, question)

Here’s a kid-friendly way to understand it.

What it does
- It goes through a list called books.
- For each book, it grabs the value of the author if there is one (and it’s not empty).
- It collects all those author names into a set (which means duplicates get squashed so you only have unique names).
- Then it hands those names out one by one to whatever is asking for them, like a conveyor belt.

Why this design
- Why a set? To automatically remove duplicates. If many books have the same author, you only get that author once.
- Why yield from? It’s like saying “please give me each author one by one” and then the outer code gets each author in turn, without building a list to return all at once.
- Why filter with “if author”? If a book has no author, we skip it instead of putting a None or empty thing into the results.

Simple diagram (idea in short)
- Books: [book1, book2, book3, book4]
  - book1 -> author A
  - book2 -> author B
  - book3 -> author A
  - book4 -> (no author)
- Set of authors: {A, B}  (duplicates removed)
- Result by yield from: A, then B (in some order)

Important notes
- The order of authors from a set isn’t guaranteed, so you might get A then B or B then A.
- The set is created before yielding, so all unique authors are collected first, which uses memory proportional to the number of unique authors.
- If you need to keep the original order and still remove duplicates, you’d change the approach (but that’s a different technique).

In short: it emits each unique author from the books, skipping any missing authors, one by one.

In [20]:
# Get gpt-4o-mini to answer, with streaming
MODEL_GPT = 'gpt-4o-mini'

answer_question(MODEL_GPT, question)



Alright! Let’s break this down into simple pieces, like building blocks. 

Imagine we have a big box filled with books, and each book has some information, like the title and the author. 

### Here’s what each part of the code does:

1. **books**: This is our big box full of books. Each book is like a little treasure, and some treasures have names (this name is the author).

2. **book.get("author")**: This is like opening each book and checking to see who wrote it. If the book has an author, we’ll see their name.

3. **if book.get("author")**: We only want to pick the books that actually have an author. If a book does not have a name, we skip it!

4. **{... for book in books}**: This is like saying, "Let’s look at every book in the box and see if we can find an author." If we find a name, we’ll keep it!

5. **yield from**: This is a special magic word that lets us say, "Here’s a way to give back the names we found!" It’s like handing out candies to friends, one by one.

### Putting it all together:

So, what this code is doing is:

- It looks into a box of books.
- It checks each book.
- If it finds an author’s name, it saves it.
- Finally, it hands out each unique author’s name one at a time!

### Picture It:

Here’s a little picture in words to help you see it:

```
[ Books Box ]
   |
   |--- Book 1: {Author: "Alice"}
   |--- Book 2: {Author: "Bob"}
   |--- Book 3: {Author: ""}
   |--- Book 4: {Author: "Alice"}

- We look in the box.
- Keep "Alice" from Book 1.
- Keep "Bob" from Book 2.
- Skip Book 3 because it has no author.
- Keep "Alice" from Book 4, but we already have her.

In the end, we end up with:
- "Alice"
- "Bob"

And that’s it! Isn’t that fun?

In [21]:
# Get Llama 3.2:1b to answer

answer_question(MODEL_LLAMA, question)

Oh boy, let me explain this code to you in a super fun way!

Imagine you have a big box of books, and inside the box, there are many books that have an author name written on the cover. This code is like a robot that wants to read the author names of these books.

Here's what it does step by step:

1. `yield from` is like a magic "pick and choose" button. It lets the code jump to the next thing it wants to do, instead of doing all of it.

2. Inside the brackets `{`, it's trying to fetch the author names of all the books that meet certain conditions. We need to check if the book has an author.
3. `book.get("author")` is like a special "get the author" button. If the book has an author, this button is pressed and the author's name gets stored in the variable "author".
4. `for book in books` is a loop that prints out the author names of all the books that have an author.

So, what does `yield from` do? It breaks down this whole process into smaller, smaller parts, each one doing its own thing, but eventually making the robot (or the code) wait for one thing to happen, and then move on.

Think of it like a team of little helpers that work together to fetch the author names:

- The first helper, "fetching the author", reads the book and prints its author name.
- The second helper, "checking if it has an author", checks that the book really has an author name.
- The third helper, "printing the author", prints the book's author name.
- The loop says, "Okay, all these helpers are done, wait for me to print all the book names!"

This makes the code super efficient and fun to read!